In [ ]:
#| echo: false
#| output: true

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

if "pivot_pct" not in globals() or "top_keywords" not in globals():
    raise RuntimeError("Run Cell 20 first to build normalized keyword trend data.")

keyword_pct_long = (
    pivot_pct.reset_index()
    .rename(columns={"index": "year"})
    .melt(id_vars="year", var_name="keyword", value_name="pct")
)
keyword_pct_long["year"] = pd.to_numeric(keyword_pct_long["year"], errors="coerce").astype(int)
keyword_pct_long["pct"] = pd.to_numeric(keyword_pct_long["pct"], errors="coerce").fillna(0.0)

keyword_pct_long["rank"] = keyword_pct_long.groupby("year")["pct"].rank(
    method="min", ascending=False
)

keywords_by_total = sorted(top_keywords, key=lambda k: float(pivot_pct[k].sum()), reverse=True)

Prepared long-form keyword trend table: 120 rows, 30 keywords, 4 years.


In [ ]:
#| echo: false
#| output: true

import re

import pandas as pd
import plotly.graph_objects as go
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
TOP_KEYWORDS = 30
PAPER_ITEM_TYPES = {"journalarticle", "manuscript", "preprint"}


def _extract_year(value):
    match = re.search(r"(19|20)\d{2}", str(value or ""))
    if not match:
        return None
    year = int(match.group(0))
    return year if 1900 <= year <= 2100 else None


def _normalize_text(value):
    if pd.isna(value):
        return ""
    return str(value).strip()


def _normalize_tag(tag):
    # Keep multi-word keywords together while normalizing whitespace/case.
    return re.sub(r"\s+", " ", _normalize_text(tag)).lower()


if "load_zotero_items" in globals():
    keyword_items = load_zotero_items()
else:
    raise RuntimeError("load_zotero_items() is not available. Run the earlier setup cell first.")

excluded_exact = set()
if "exclude_terms" in globals():
    excluded_exact.update(_normalize_tag(term) for term in exclude_terms)

rows = []
for item in keyword_items:
    data = item.get("data", {})
    item_type = str(data.get("itemType", "")).strip().lower()
    if item_type not in PAPER_ITEM_TYPES:
        continue

    year = _extract_year(data.get("date", ""))
    if year is None:
        continue

    tags_raw = [
        _normalize_tag(t.get("tag", ""))
        for t in data.get("tags", [])
        if _normalize_tag(t.get("tag", ""))
    ]

    # De-duplicate repeated tags within a paper.
    tags_unique = sorted(set(tags_raw))

    filtered_tags = []
    for tag in tags_unique:
        if tag in excluded_exact:
            continue

        words = [w for w in re.findall(r"[a-zA-Z]+", tag) if w]
        if not words:
            continue
        if all(w in ENGLISH_STOP_WORDS for w in words):
            continue

        filtered_tags.append(tag)

    if not filtered_tags:
        continue

    for tag in filtered_tags:
        rows.append({"year": int(year), "keyword": tag, "count": 1})

df_kw_year = pd.DataFrame(rows)
if df_kw_year.empty:
    print("Warning: no year-tagged paper keywords found for keyword trend chart.")
else:
    keyword_totals = (
        df_kw_year.groupby("keyword", as_index=False)["count"]
        .sum()
        .sort_values(["count", "keyword"], ascending=[False, True])
    )
    top_keywords = keyword_totals.head(TOP_KEYWORDS)["keyword"].tolist()

    yearly = (
        df_kw_year[df_kw_year["keyword"].isin(top_keywords)]
        .groupby(["year", "keyword"], as_index=False)["count"]
        .sum()
        .sort_values(["year", "keyword"])
    )

    # Preserve the full year range from all tagged paper records.
    min_year = int(df_kw_year["year"].min())
    max_year = int(df_kw_year["year"].max())
    all_years = list(range(min_year, max_year + 1))

    pivot = (
        yearly.pivot(index="year", columns="keyword", values="count")
        .reindex(all_years, fill_value=0)
        .sort_index()
    )

    # Normalize each year so stacked values represent percentages of keyword uses.
    row_sums = pivot.sum(axis=1)
    pivot_pct = pivot.div(row_sums.where(row_sums > 0, 1), axis=0) * 100

    fig = go.Figure()
    for kw in top_keywords:
        if kw not in pivot_pct.columns:
            continue
        fig.add_trace(
            go.Scatter(
                x=pivot_pct.index.tolist(),
                y=pivot_pct[kw].tolist(),
                mode="lines",
                stackgroup="one",
                name=kw,
                hovertemplate="%{x}<br>%{fullData.name}: %{y:.1f}%<extra></extra>",
            )
        )

    fig.update_layout(
        title="Stacked Keyword Usage by Year (Top Paper Keywords)",
        xaxis_title="Year",
        yaxis_title="Keyword uses (%)",
        height=560,
        legend_title="Keyword",
    )
    fig.update_xaxes(tickmode="linear", dtick=1)
    fig.update_yaxes(range=[0, 100], ticksuffix="%")
    fig.show()

    print(
        f"Plotted normalized stacked keyword trends for {len(top_keywords)} keywords across "
        f"{len(pivot_pct.index)} years ({min_year}-{max_year})."
    )

Loaded Zotero items from API: 126 records


Plotted normalized stacked keyword trends for 30 keywords across 4 years (2023-2026).


In [ ]:
#| echo: false
#| output: true

heatmap_data = pivot_pct[keywords_by_total].T

fig = px.imshow(
    heatmap_data,
    labels={"x": "Year", "y": "Keyword", "color": "% of keyword uses"},
    title="Keyword Trend Heatmap (Normalized by Year)",
    aspect="auto",
    color_continuous_scale="YlGnBu",
)
fig.update_layout(height=max(500, 24 * len(heatmap_data) + 180))
fig.show()

In [ ]:
#| echo: false
#| output: true

small_mult_data = keyword_pct_long[keyword_pct_long["keyword"].isin(keywords_by_total)].copy()

fig = px.line(
    small_mult_data,
    x="year",
    y="pct",
    facet_col="keyword",
    facet_col_wrap=5,
    markers=True,
    title="Small Multiples: Keyword Trends by Year",
)
fig.update_yaxes(matches=None, ticksuffix="%")
fig.update_xaxes(dtick=1)
fig.update_layout(height=max(900, 170 * ((len(keywords_by_total) + 4) // 5)))
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.show()

In [ ]:
#| echo: false
#| output: true

TOP_BUMP = 15
bump_keywords = keywords_by_total[:TOP_BUMP]
bump_data = keyword_pct_long[keyword_pct_long["keyword"].isin(bump_keywords)].copy()

fig = px.line(
    bump_data,
    x="year",
    y="rank",
    color="keyword",
    markers=True,
    title="Bump Chart: Keyword Rank by Year",
)
fig.update_yaxes(
    autorange="reversed",
    dtick=1,
    title="Rank (1 = highest share)",
)
fig.update_xaxes(dtick=1)
fig.update_layout(height=620)
fig.show()

In [ ]:
#| echo: false
#| output: true

TOP_SLOPE = 15
start_year = int(pivot_pct.index.min())
end_year = int(pivot_pct.index.max())

slope_df = pd.DataFrame({
    "keyword": pivot_pct.columns,
    "start_pct": pivot_pct.loc[start_year].values,
    "end_pct": pivot_pct.loc[end_year].values,
})
slope_df["delta"] = slope_df["end_pct"] - slope_df["start_pct"]
slope_df = slope_df.reindex(
    slope_df["delta"].abs().sort_values(ascending=False).index
).head(TOP_SLOPE)

fig = go.Figure()
for _, row in slope_df.iterrows():
    fig.add_trace(
        go.Scatter(
            x=[start_year, end_year],
            y=[row["start_pct"], row["end_pct"]],
            mode="lines+markers",
            name=row["keyword"],
            hovertemplate=(
                "%{x}<br>"
                + row["keyword"]
                + ": %{y:.1f}%<extra></extra>"
            ),
            showlegend=False,
        )
    )

fig.add_trace(
    go.Scatter(
        x=[None],
        y=[None],
        mode="markers",
        marker=dict(color="rgba(0,0,0,0)"),
        showlegend=False,
        hoverinfo="skip",
    )
)

fig.update_layout(
    title=f"Slope Chart: Keyword Share Change ({start_year} to {end_year})",
    xaxis_title="Year",
    yaxis_title="% of keyword uses",
    height=620,
)
fig.update_xaxes(tickmode="array", tickvals=[start_year, end_year])
fig.update_yaxes(ticksuffix="%")
fig.show()

In [ ]:
#| echo: false
#| output: true

stream_data = keyword_pct_long[keyword_pct_long["keyword"].isin(keywords_by_total)].copy()
stream_data = stream_data.sort_values(["year", "keyword"])

fig = px.area(
    stream_data,
    x="year",
    y="pct",
    color="keyword",
    line_group="keyword",
    title="Streamgraph-Style Area: Keyword Shares by Year",
    labels={"pct": "% of keyword uses", "year": "Year"},
)
fig.update_xaxes(dtick=1)
fig.update_yaxes(ticksuffix="%")
fig.update_layout(height=620)
fig.show()

In [ ]:
#| echo: false
#| output: true

TOP_PER_YEAR = 8
bar_data = (
    keyword_pct_long.sort_values(["year", "pct"], ascending=[True, False])
    .groupby("year", as_index=False)
    .head(TOP_PER_YEAR)
)

fig = px.bar(
    bar_data,
    x="year",
    y="pct",
    color="keyword",
    barmode="group",
    title=f"Grouped Bars: Top {TOP_PER_YEAR} Keywords per Year",
    labels={"pct": "% of keyword uses", "year": "Year"},
)
fig.update_xaxes(dtick=1)
fig.update_yaxes(ticksuffix="%")
fig.update_layout(height=560)
fig.show()